#<font color="Green">**Notebook Purpose**</font>

This notebook documents the transformation of raw query outputs into four standardized CSV files, each containing cleaned and structured data used in subsequent stages of analysis.

The four csv files will be as follows:


1.   **`patient_demographics.csv:`** This file will  hold demographic information about the patients.
        

2.   **`medication_info.csv:`** The goal here is to create a file that has the names (cls.short) of the medications patients took as well as some supplementary info regarding those medications. Currently, the raw medication_info file has the medications represented by codes instead of names.


3.   **`lab_results.csv:`** This file will contain information on HbA1c test results for each patient.

4.   **`vital_signs.csv:`** This file was intended to hold information on both BMI and weight data for each patient, but as demonstrated in the analysis below, issues with the weight data meant that we chose to move forward with only the BMI data.


---


###<font color ="Red"> Required Data </font>

To run the code blocks in this notebook, you will need the four CSVs that are a result of the queries outlined in the 'Cohort Creation Queries' document found in the 'Queries' folder. They are of the same name as the files to be outputted.

Additionally, a file called 'Ingredient Translation File' will be needed. You can find this file in the 'Supplementary Data' folder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

##<font color="black">**Read In Data**</font>



In [ ]:
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
medication_info = pd.read_csv('/content/medication_info.csv', dtype={'code': str}) # In our data there exists a code with str representation
lab_results = pd.read_csv('/content/lab_results.csv')
vital_signs = pd.read_csv('/content/vital_signs.csv')

ingredient_translation = pd.read_excel('/content/Ingredient Translation File.xlsx')

In [ ]:
len(patient_demographics['patient_id'].unique())

In [ ]:
patient_demographics.head()

In [ ]:
len(medication_info['patient_id'].unique())

In [ ]:
medication_info.head()

In [ ]:
len(lab_results['patient_id'].unique())

In [ ]:
lab_results.head()

In [ ]:
len(vital_signs['patient_id'].unique())

In [ ]:
vital_signs.head()

In [ ]:
ingredient_translation.head()

##<font color="black">**Creating patient_demographics**</font>

The only preprocessing required for the patient_demographics file is the creation of a standardized race/ethnicity column that appropriately incorporates patients identified as Hispanic.

In [ ]:
print(patient_demographics['ethnicity'].unique())

In [ ]:
print(patient_demographics['race'].unique())

In [ ]:
hispanic_df = patient_demographics[patient_demographics['ethnicity'] == 'Hispanic or Latino']

hispanic_with_race = hispanic_df[hispanic_df['race'].notna() & (hispanic_df['race'] != '')]

race_counts = hispanic_with_race['race'].value_counts(dropna=False)

print("Total number of Hispanic or Latino patients:", len(hispanic_df))
print("Number of Hispanic or Latino patients with a known race:", len(hispanic_with_race))
print("\nRace distribution among Hispanic or Latino patients:")
print(race_counts)

Next, we create a combined race/ethnicity column by assigning “Hispanic” to all patients with Hispanic ethnicity; for all others, we use their recorded race.

In [ ]:
patient_demographics['race/ethnicity'] = np.where(
    patient_demographics['ethnicity'] == 'Hispanic or Latino',
    'Hispanic',
    patient_demographics['race'])

patient_demographics.head()

For the final step, we can shorten the description for some of the values in the race column. I will also combine 'Native Hawaiian or Other Pacific Islander' and 'American Indian or Alaska Native' into the 'Other' classifier.

In [ ]:
patient_demographics['race/ethnicity'] = patient_demographics['race/ethnicity'].replace({
    'Black or African American': 'Black',
    'Other Race': 'Other',
    'Native Hawaiian or Other Pacific Islander': 'Other',
    'American Indian or Alaska Native': 'Other',
    'Unknown' : 'Other'
})

patient_demographics['race/ethnicity'].value_counts()

##<font color="black">**Creating medication_info**</font>




###<font color="black">**Step 1: Check Overlap**</font>

In this step I check to see what medications are represented in medication_info and ingredient_translation. If there are any missing/unknown medication ingredients, I manually import them.

I will start by looking at what is held in the code_description column in medication_ingredients and the ing.name column in ingredient_translation.

In [ ]:
medication_info_ingredients = medication_info['code_description'].unique()
ingredient_translation_ingredients = ingredient_translation['ing.name'].unique()

In [ ]:
print("=== medication_info_ingredients ===")
for i, item in enumerate(medication_info_ingredients):
    print(f"{i+1}. {item}")

print("\n=== ingrediant_translation_ingredients ===")
for i, item in enumerate(ingredient_translation_ingredients):
    print(f"{i+1}. {item}")

The ingredient_translation file seems to be missing 4 ingredients that are clearly listed in the excel file. I will add them manually here.

In [ ]:
new_rows = pd.DataFrame([
    {
        'ing.name': 'bromocriptine',
        'cls.long': 'Dopamine agonists',
        'cls.short': 'Other',
        'notes': ''
    },
    {
        'ing.name': 'colesevelam',
        'cls.long': 'Bile acid sequestrants',
        'cls.short': 'Other',
        'notes': ''
    },
    {
        'ing.name': 'colesevelam hydrochloride',
        'cls.long': 'Bile acid sequestrants',
        'cls.short': 'Other',
        'notes': ''
    },
    {
        'ing.name': 'guar gum',
        'cls.long': 'Other blood glucose lowering drugs, excl. insulins',
        'cls.short': 'Other',
        'notes': ''
    }
])

ingredient_translation = pd.concat([ingredient_translation, new_rows], ignore_index=True)

In [ ]:
ingredient_translation_ingredients = ingredient_translation['ing.name'].unique()
print("=== ingrediant_translation_ingredients ===")
for i, item in enumerate(ingredient_translation_ingredients):
    print(f"{i+1}. {item}")

Some of the ingredients in medication_info_ingredients have words seperated by commas so I will normalize those values.

In [ ]:
medication_info['code_description'] = medication_info['code_description'].str.replace(',', '')
medication_info['code_description'] = medication_info['code_description'].str.replace('  ', ' ')
medication_info_ingredients = medication_info['code_description'].unique()

Now to check for overlap between medication_info_ingredients and
ingredient_translation_ingredients

In [ ]:
medication_ii_set = set(medication_info_ingredients)
ingredient_ti_set = set(ingredient_translation_ingredients)

missing_ingredients = medication_ii_set - ingredient_ti_set

In [ ]:
print(f"{len(missing_ingredients)} ingredients are missing:")
for ingredient in sorted(missing_ingredients):
    print(ingredient)

I will now manually add the three missing ingredients to the ingredient_translation file.

In [ ]:
new_rows = pd.DataFrame([
    {
        'ing.name': 'BEXAGLIFLOZIN (deprecated 2023)',
        'cls.long': 'Sodium-glucose co-transporter 2 (SGLT2) inhibitors',
        'cls.short': 'SGLT2',
        'notes': ''
    },
    {
        'ing.name': 'bexagliflozin',
        'cls.long': 'Sodium-glucose co-transporter 2 (SGLT2) inhibitors',
        'cls.short': 'SGLT2',
        'notes': ''
    },
    {
        'ing.name': 'sotagliflozin',
        'cls.long': 'Sodium-glucose co-transporter 2 (SGLT2) inhibitors',
        'cls.short': 'SGLT2',
        'notes': ''
    }
])

ingredient_translation = pd.concat([ingredient_translation, new_rows], ignore_index=True)

Now to double check that everything is covered.

In [ ]:
ingredient_translation_ingredients = ingredient_translation['ing.name'].unique()

ingredient_ti_set = set(ingredient_translation_ingredients)

missing_ingredients = medication_ii_set - ingredient_ti_set
print(f"{len(missing_ingredients)} ingredients are missing:")
for ingredient in sorted(missing_ingredients):
    print(ingredient)

###<font color="black">**Step 2: Combine Table Information**</font>

I will now join the tables on the code_descrition column from medication_info and the ing.name column from the ingredient_translation table.

In [ ]:
medication_info = medication_info.merge(
    ingredient_translation[['ing.name', 'cls.short']],
    left_on='code_description',
    right_on='ing.name',
    how='left'
)

In [ ]:
medication_info.head()

###<font color="black">**Step 3: Table Cleaning**</font>

Here I will drop some unnecessary columns and rename a few for clarity's sake.

Additionally, I will reclassify the medication class 'Alpha-GI' to be grouped with 'Other' and 'f-INS' will be grouped with 'Insulin'. This is consistent with our plan outlined in the methods section of the paper.

In [ ]:
medication_info = medication_info.drop(columns=['ing.name', 'code'])
medication_info = medication_info.rename(columns={'cls.short': 'medication_class'})
medication_info = medication_info.rename(columns={'code_description': 'ingredient'})
medication_info.head()

In [ ]:
classes = medication_info['medication_class'].unique()
print(classes)

In [ ]:
medication_info['medication_class'] = medication_info['medication_class'].replace('Alpha-GI', 'Other')
medication_info['medication_class'] = medication_info['medication_class'].replace('f-INS', 'Insulin')
new_classes = medication_info['medication_class'].unique()
print(new_classes)

##<font color="black">**Creating lab_results**</font>

This will just involve dropping and renaming some tables.

In [ ]:
lab_results.head()

The code information is not required for our analysis and has been removed, but the relevant LOINC codes are listed below for reference.

In [ ]:
lab_results['code'].unique().tolist()

In [ ]:
lab_results = lab_results.drop(columns=['encounter_id', 'code_system', 'code', 'path'])
lab_results = lab_results.rename(columns={'code_description': 'lab_test'})
lab_results.head()

In [ ]:
lab_results['lab_result_text_val'].unique().tolist()

Since all text values are NaN, I will drop that column as well.

In [ ]:
lab_results = lab_results.drop(columns=['lab_result_text_val'])
lab_results.head()

##<font color="black">**Creating vital_signs**</font>

This will just involve dropping and renaming some tables.

In [ ]:
vital_signs.head()

In [ ]:
vital_signs = vital_signs.drop(columns=['path'])
vital_signs = vital_signs.rename(columns={'code_description': 'vital_sign'})
vital_signs.head()

In [ ]:
vital_signs['text_value'].unique().tolist()

Since all text values are NaN, I will drop that column as well.

In [ ]:
vital_signs = vital_signs.drop(columns=['text_value'])
vital_signs.head()

##<font color="black">**Cohort Criteria Verification**</font>

In this section, we verify that the patient data satisfy the predefined inclusion and exclusion criteria. These criteria should already be met, as they were applied during the database query process.

Certain criteria cannot be validated within this Python environment; these items are highlighted in red. Details on how these criteria were assessed can be found in the document containing our query statements, located in the Queries folder.

As an additional note, all medications included in the dataset are assumed to be glucose-lowering medications. The query logic used to construct standardized_terminology_glm provides further justification for this assumption.

**Inclusion:**
1. <font color="red">patients with T2D diagnosis with diagnosis dates between 1-1-2019 and 7-1-2019 </font>
2. <font color="red">patients with any encounter in second half of 2024 (after 7-1-2024)</font>
3. Encounter with glm prescribed in the first half of 2019
4. <font color="red">Must have an encounter before 2018. </font>
5. Older than 18+ at the moment of 2019H1 (As well as no death date before December 31st 2024)

**Exclusion:**
1. No encounter with glm before 2019
2. <font color="red">No patients with two HbA1c lab result above 6.5% before July 1st, 2018. (LOINC) </font>
3. <font color="red"> No patients with Diagnosis of diabetes before 2019. </font>
4. No patients who are from Ex-US or Other
5. <font color="red"> No patients who were pregnant during our trial period. </font>

Before looking at the criteria, lets just make sure that all our data is within our study range (2019-2024).

In [ ]:
medication_info['start_date'] = pd.to_datetime(medication_info['start_date'])
print('Earliest prescription date is:', medication_info['start_date'].min())
print('Oldest prescription date is:', medication_info['start_date'].max())

We will start by verifying inclusion criteria 3.

3. Encounter with glm prescribed in the first half of 2019

In [ ]:
mask_2019 = medication_info['start_date'] < '2019-07-01'

patients_2019 = set(medication_info[mask_2019]['patient_id'].unique())

all_patients = set(medication_info['patient_id'].unique())

patients_missing_criteria3 = all_patients - (patients_2019)

print(f"Number of patients missing criteria 3: {len(patients_missing_criteria3)}")

Now we can verify inclusion criteria 5.

5. Older than 18+ at the moment of 2019H1 (As well as no death date before December 31st 2024)

We will split this up into two parts. First check for people younger than 18 at moment of 2019H1, then check for proper death criteria.

In [ ]:
mask_18plus = patient_demographics['year_of_birth'] <= 2000

patients_18plus = set(patient_demographics[mask_18plus]['patient_id'].unique())

all_patients = set(patient_demographics['patient_id'].unique())

patients_missing_criteria5 = all_patients - (patients_18plus)

print(f"Number of patients younger than 18 as of 2019: {len(patients_missing_criteria5)}")

In [ ]:
patient_demographics['month_year_death'].unique().tolist()

Since there are no death dates that occur before Decemeber 31st 2024, this part of the criteria is also fufilled.

Now I will look at exclusion criteria 1.

1. No encounter with glm before 2019.

This is actually already verified when we checked our date ranges in the first code block of this section, seeing as no prescriptions of any glm are recorded before 2019.

Next, I will verfiy exclusion criteria 4.

4. No patients who are from Ex-US or Other

This exclusion criterion was not applied during the initial queries. Accordingly, we will identify all patients with an Ex-US or “Other” location designation and remove their patient IDs from all relevant files.

In [ ]:
patient_demographics['patient_regional_location'].unique().tolist()

In [ ]:
EX_US_pis = set(patient_demographics[patient_demographics['patient_regional_location'] == 'Ex-US']['patient_id'])
Unknown_pis = set(patient_demographics[patient_demographics['patient_regional_location'] == 'Unknown']['patient_id'])
combined_pis = EX_US_pis.union(Unknown_pis)
print(f"Number of patients from Ex-US or Unknown: {len(combined_pis)}")

In [ ]:
filtered_patient_demographics = patient_demographics[~patient_demographics['patient_id'].isin(combined_pis)]
filtered_medication_info = medication_info[~medication_info['patient_id'].isin(combined_pis)]
filtered_lab_results = lab_results[~lab_results['patient_id'].isin(combined_pis)]
filtered_vital_signs = vital_signs[~vital_signs['patient_id'].isin(combined_pis)]

In [ ]:
print('Demographics:', len(filtered_patient_demographics['patient_id'].unique()))
print('Medications:', len(filtered_medication_info['patient_id'].unique()))
print('Lab Results:', len(filtered_lab_results['patient_id'].unique()))
print('Vital Signs:', len(filtered_vital_signs['patient_id'].unique()))

As expected, the lab results and vital signs tables contain fewer unique patients, as not all individuals have corresponding measurements recorded. The key requirement is that the demographics and medication tables contain the same set of patients.

We now verify that patient membership is consistent across these tables.

In [ ]:
print(set(filtered_patient_demographics['patient_id'].unique()) == set(filtered_medication_info['patient_id'].unique()))
print((set(filtered_lab_results['patient_id'].unique()) - set(filtered_patient_demographics['patient_id'].unique())) == set())
print((set(filtered_vital_signs['patient_id'].unique()) - set(filtered_patient_demographics['patient_id'].unique())) == set())

##<font color="black">**More Robust Data Cleaning**</font>

Up to this point, only basic data cleaning has been performed. Additional steps, such as outlier detection and further validation checks, are still required before exporting the final CSV files.

To begin this process, I will examine the distributions and value ranges of all numeric variables in the patient demographics and medication datasets.

In [ ]:
# Looking at distribution of year of birth
filtered_patient_demographics['year_of_birth'].hist()

In [ ]:
# Duplicate entry removal
print('Duplicate entries:', filtered_medication_info.duplicated().sum())
filtered_medication_info = filtered_medication_info.drop_duplicates()
filtered_medication_info.shape

In [ ]:
print('Duplicate entries:', filtered_patient_demographics.duplicated().sum())

####<font color="black">**Data Cleaning for Lab Results**</font>

The first step is to remove any unnecessary columns and apply standardized naming to the remaining variables.

In [ ]:
filtered_lab_results.head()

In [ ]:
print(filtered_lab_results['lab_test'].unique())

I will check for any duplicate entries and any NaN values and remove those entries.

In [ ]:
print('Duplicate entries:', filtered_lab_results.duplicated().sum())
filtered_lab_results = filtered_lab_results.drop_duplicates()
print('NaN values:', filtered_lab_results['lab_result_num_val'].isna().sum())
filtered_lab_results = filtered_lab_results.dropna(subset=['lab_result_num_val'])
filtered_lab_results.shape

Since the  three lab tests are all [NGSP certified](https://ngsp.org/docs/methods.pdf?utm) and therefore traceable to the [DCCT reference method](https://pubmed.ncbi.nlm.nih.gov/8366922/), we can combine their results for analysis.

Now to look at the distribution of the lab results.

In [ ]:
filtered_lab_results['lab_result_num_val'].describe()

The results indicate that there are clear outliers. While the FDA states the measuring range of the devices used to measure HbA1C is 3.5% to 20%. ([Source, page 9](https://www.accessdata.fda.gov/cdrh_docs/reviews/K151321.pdf?utm_source=chatgpt.com)), I will expand that range up to 50% in case there any extreme outliers detected.

In [ ]:
val = pd.to_numeric(filtered_lab_results['lab_result_num_val'], errors='coerce')

# define outliers: < 3.5 or >= 50 (and also NaN -> outlier if you want)
invalid_mask = val.isna() | (val < 3.5) | (val >= 50)

# keep only valid rows
HbA1C_outliers = filtered_lab_results.loc[invalid_mask].copy() # for auditing
filtered_lab_results = filtered_lab_results.loc[~invalid_mask].copy()

print(f"Removed rows: {len(HbA1C_outliers):,}")
print(f"Remaining rows: {len(filtered_lab_results):,}")


In [ ]:
filtered_lab_results['lab_result_num_val'].describe()

In [ ]:
plt.figure(figsize=(10, 6))

filtered_lab_results['lab_result_num_val'].hist(bins=50)

plt.title("Distribution of Lab Result Values", fontsize=14)
plt.xlabel("Lab Result Value (%)", fontsize=12)
plt.ylabel("Number of Records", fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

The following code block evaluates how the mean changes as the upper outlier threshold is progressively lowered. The results demonstrate that trimming extremely high HbA1c values has minimal impact on the overall mean.

In [ ]:
temp_data = filtered_lab_results[filtered_lab_results['lab_result_num_val'] <= 20]
print('Mean after trim:', temp_data['lab_result_num_val'].mean())

####<font color="black">**Data Cleaning for Vital Signs**</font>

There are three different types of vital signs recorded. Here I will look at the values recorded for those vital signs and filter out outliers/erroneous data.

First thing I will do is look for any duplicates or NaN values and remove those entries.

In [ ]:
filtered_vital_signs.head()

In [ ]:
print('Duplicate entries:', filtered_vital_signs.duplicated().sum())
filtered_vital_signs = filtered_vital_signs.drop_duplicates()
print('NaN values:', filtered_vital_signs['value'].isna().sum())
filtered_vital_signs = filtered_vital_signs.dropna(subset=['value'])
filtered_vital_signs.shape

Now I will look at the three vital signs present in the data set.

In [ ]:
filtered_vital_signs['vital_sign'].unique().tolist()

“Weight” and “Body Weight” appear to represent the same measurement. To determine whether any differences exist, we will examine their associated codes and units of measure.

In [ ]:
weight_df = filtered_vital_signs[filtered_vital_signs['vital_sign'] == 'Weight']
body_weight_df = filtered_vital_signs[filtered_vital_signs['vital_sign'] == 'Body weight']

In [ ]:
print('===== Weight =====')
print(weight_df['code'].unique().tolist())
print(weight_df['units_of_measure'].unique().tolist())

print(' ')

print('===== Body Weight =====')
print(body_weight_df['code'].unique().tolist())
print(body_weight_df['units_of_measure'].unique().tolist())

In [ ]:
weight_df['value'].hist()

In [ ]:
body_weight_df['value'].hist()

Both variables use the same units but are associated with different codes. Upon reviewing these codes, we confirmed that they are intended to measure the same construct.

To verify this empirically, we will create a data frame comparing the values of both vital signs for patients who have measurements for each on the same date. Ideally, these values should be equivalent.

<font color=red>Note:</font> This df does not include patients who only have a recorded value for one of the vital signs.

In [ ]:
filtered_vital_signs['date'] = pd.to_datetime(filtered_vital_signs['date'])

# 1. Create two separate DataFrames (retain 'encounter_id')
df_weight = filtered_vital_signs[filtered_vital_signs['vital_sign'] == 'Weight'][[
    'patient_id', 'date', 'encounter_id', 'value', 'code'
]]
df_body_weight = filtered_vital_signs[filtered_vital_signs['vital_sign'] == 'Body weight'][[
    'patient_id', 'date', 'encounter_id', 'value', 'code'
]]

# 2. Rename columns for clarity
df_weight = df_weight.rename(columns={
    'value': 'weight_value',
    'code': 'weight_code',
    'encounter_id': 'weight_encounter_id'
})
df_body_weight = df_body_weight.rename(columns={
    'value': 'body_weight_value',
    'code': 'body_weight_code',
    'encounter_id': 'body_weight_encounter_id'
})

# 3. Merge on patient_id and date
merged = pd.merge(df_weight, df_body_weight, on=['patient_id', 'date'], how='inner')

# 4. Compare values (with ±100 lb tolerance)
merged['value_match'] = (merged['weight_value'] - merged['body_weight_value']).abs() <= 100

# 5. Identify mismatches
mismatches = merged[~merged['value_match']].copy()

# 6. Summary
print(f"Total overlapping records: {len(merged)}")
print(f"Mismatches found: {len(mismatches)}")

mismatches.head()

<font color=red>IMPORTANT!</font>

It is apparent that the 'weight' and 'body_weight' variables do not share the same units, despite their code discriptions indicating that they should.

Next, I will dive deeper into this data to better understand how the codes might be distributed.

In [ ]:
print('Total Entries in vital_signs: ', len(filtered_vital_signs))
print('LOINC 3141-9 Code Count: ', len(weight_df))
print('LOINC 29463-7 Code Count: ', len(body_weight_df))
print('BMI Vital Sign Count: ', len(filtered_vital_signs[filtered_vital_signs['vital_sign'] == 'Body Mass Index']))

In [ ]:
weight_df['value'].describe()

In [ ]:
body_weight_df['value'].describe()

Based on this initial analysis, there are substantially more observations associated with LOINC code 29463-7 (Body Weight) than with LOINC code 3141-9 (Weight). Despite this difference in frequency, both variables exhibit highly similar descriptive statistics, making it difficult to distinguish values from one code versus the other based on distribution alone.

Attempts to reconcile the two variables—such as converting between kilograms and pounds—did not yield consistent or interpretable results. In addition, further review of the TriNetX data dictionary indicated inconsistencies that prevent reliable use of these measurements. For this reason, we determined that the weight data could not be used in a meaningful or technically sound manner and notified TriNetX of the issue.

Following the evaluation of the BMI data, and assuming no comparable inconsistencies are detected, we will proceed using BMI as the primary vital sign for subsequent analyses.


Now lets take a look at BMI.

In [ ]:
filtered_vital_signs[filtered_vital_signs['vital_sign'] == 'Body Mass Index']['value'].describe()

Since the max recorded BMI isn't absurdely high, I don't need to filter for outliers in that range. However, the min value is 0 which indicates possible mis-inputs for on the lower end of things. I will filter out patients who have a BMI < 10.

In [ ]:
val = pd.to_numeric(filtered_vital_signs["value"], errors="coerce")

# Bad rows = BMI entries with value <= 10
bad_bmi_mask = (filtered_vital_signs["vital_sign"] == "Body Mass Index") & (val <= 10)

# inspect the outliers
bmi_outliers = filtered_vital_signs.loc[bad_bmi_mask].copy()
print(f"Bad BMI rows: {len(bmi_outliers):,}  |  Patients affected: {bmi_outliers['patient_id'].nunique():,}")

# Keep all good rows (do NOT drop whole patients)
filtered_vital_signs = filtered_vital_signs.loc[~bad_bmi_mask].copy()
print(f"Remaining rows: {len(filtered_vital_signs):,}")

In [ ]:
filtered_vital_signs[filtered_vital_signs['vital_sign'] == 'Body Mass Index']['value'].hist()

In [ ]:
filtered_BMI_vital_signs = filtered_vital_signs[filtered_vital_signs['vital_sign'] == 'Body Mass Index']
filtered_BMI_vital_signs.head()

##<font color="black">**Exporting as CSVs**</font>

Now I will export the four files. As of 11-19-2025, those files are:



1. patient_demographics (from filtered_patient_demographics)

2. medication_info (from filtered_medication_info)   

3. lab_results (from filtered_lab_results)

4. BMI_vital_signs (from filtered_BMI_vital_signs)

In [ ]:
filtered_patient_demographics.to_csv('Cpatient_demographics.csv', index=False)
filtered_medication_info.to_csv('Cmedication_info.csv', index=False)
filtered_lab_results.to_csv('Clab_results.csv', index=False)
filtered_BMI_vital_signs.to_csv('CBMI_vital_signs.csv', index=False)